# Notebook 2 — Mask R-CNN R50-FPN (two-stage, CNN)

**Model:** Mask R-CNN ResNet-50-FPN v2 dari torchvision, pretrained COCO (`MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1`).
**Keluarga:** two-stage, CNN. Tahap 1 (RPN) mengusulkan region, tahap 2 mengklasifikasi tiap RoI dan memprediksi
mask pada **grid 28×28** per RoI yang kemudian di-*resize* ke ukuran box.

torchvision dipakai (bukan Detectron2) karena tidak perlu kompilasi saat instalasi di Kaggle. Arsitekturnya sama:
RoIAlign 14×14 → mask head → mask 28×28.

Resep: SGD (lr 0.005 untuk batch 4, momentum 0.9, weight decay 1e-4), warmup linear, lr turun ×0.1 pada 70% dan 90%
iterasi, multi-scale training (sisi pendek 0.75–1× ukuran input), horizontal flip, mixed precision fp16.
Validasi setiap `EVAL_EVERY` epoch; bobot dengan mask AP validasi tertinggi disimpan sebagai `best.pth`.

**Cara menjalankan di Kaggle:**
1. *Settings* → *Accelerator*: **GPU T4 x2**, *Internet*: **On**.
2. *Add Input* → *Notebooks* → pilih output **Notebook 0** (berisi `indoor_seg_data.zip`).
3. **Pilot dulu** (`PILOT = True`, jalankan interaktif): 2 epoch pada data 100% untuk mengukur waktu per epoch.
   Notebook mencetak saran `EPOCHS` agar run 100% selesai dalam `TARGET_HOURS_100` jam.
4. Isi `EPOCHS`, set `PILOT = False`, lalu **Save Version → Save & Run All** (berjalan di background, maks 12 jam).

**Cara kerja dengan 2 GPU:** setiap run memakai satu GPU dan berjalan sebagai proses terpisah. GPU0 menjalankan run 100%,
GPU1 menjalankan run 25% lalu 50%. Batch size, learning rate, dan jumlah epoch identik untuk ketiga fraksi, jadi
satu-satunya yang berbeda antar run adalah jumlah data latih.

**Evaluasi:** semua model dievaluasi dengan fungsi pycocotools yang sama (`seg_common.py`, identik di Notebook 1–3)
pada split `valid`, `test`, dan `test_own` (kalau sudah ada), maksimal 100 prediksi per gambar.

**Output** (`/kaggle/working`): `runs/<model>_<fraksi>/` (bobot, `results.csv`, prediksi, evaluasi),
`metrics_<model>.csv`, `per_class_<model>.csv`, `gpu_time_log.csv`, dan `logs/`.

**Test set buatan sendiri yang datang belakangan:** jalankan ulang Notebook 0 dengan `OWN_TEST_*` terisi, lalu
jalankan notebook ini dengan `RUN_TRAINING = False` dan output versi training notebook ini ditambahkan sebagai input.
Bobot lama dipakai, tidak ada training ulang.

In [ ]:
!pip install -q pycocotools

## 1. Konfigurasi

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
import yaml

MODEL_KEY = "mask_rcnn_r50"
MODEL_LABEL = "Mask R-CNN R50-FPN v2"
FAMILY = "two-stage, CNN"

EPOCHS = 70          # pilot: 2 epoch = 3,3 mnt (~1,6 mnt/epoch, eval tiap epoch) -> saran 73; dipakai 70
BATCH = 4            # per GPU (satu run = satu GPU)
LR = 0.005
IMGSZ = 640
WORKERS = 2
EVAL_EVERY = 2

TAGS = ["025", "050", "100"]          # fraksi data latih (NRP digit 3)
SEED = 42
MAX_HOURS_PER_RUN = 3.0               # pengaman: run berhenti rapi setelah epoch yang melewati batas ini

PILOT = False                         # True: 2 epoch pada data 100% untuk mengukur waktu per epoch (sudah dijalankan)
PILOT_EPOCHS = 2
TARGET_HOURS_100 = 2.0                # target durasi run 100% untuk saran jumlah epoch
RUN_TRAINING = False                  # False: lewati training, pakai bobot dari output versi sebelumnya (training sudah selesai)

DATA_ZIP_GLOB = "/kaggle/input/**/indoor_seg_data.zip"
PREV_RUNS_GLOB = "/kaggle/input/**/runs"
DATA_ROOT = Path("/tmp/data")
WORK = Path("/kaggle/working")
RUNS = WORK / "runs"
POLL_SECONDS = 120                    # interval laporan progres

## 2. Data dari Notebook 0

In [ ]:
zips = sorted(glob.glob(DATA_ZIP_GLOB, recursive=True))
assert zips, "indoor_seg_data.zip tidak ditemukan. Tambahkan output Notebook 0 lewat Add Input."
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(DATA_ROOT)

# path absolut di YAML YOLO disamakan dengan lokasi ekstrak
for y in DATA_ROOT.glob("data_*.yaml"):
    cfg = yaml.safe_load(y.read_text(encoding="utf-8"))
    cfg["path"] = DATA_ROOT.as_posix()
    y.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")

INFO = json.loads((DATA_ROOT / "dataset_info.json").read_text(encoding="utf-8"))
EVAL_SPLITS = [s for s in ["valid", "test", "test_own"] if (DATA_ROOT / "annotations" / f"{s}.json").exists()]
print("zip    :", zips[0])
print("kelas  :", [c["name"] for c in INFO["classes"]])
print("subset :", INFO["subsets"])
print("evaluasi pada:", EVAL_SPLITS)
if "test_own" not in EVAL_SPLITS:
    print("-> test_own belum ada. Setelah Notebook 0 dijalankan ulang dengan test set buatanmu, "
          "jalankan notebook ini dengan RUN_TRAINING = False.")

## 3. Kode training dan evaluasi

Ditulis ke file supaya setiap run berjalan sebagai proses terpisah di GPU-nya sendiri. `seg_common.py` identik di Notebook 1–3.

In [ ]:
%%writefile seg_common.py
"""Fungsi bersama ketiga notebook training: seed, encode mask, evaluasi COCO (pycocotools), log CSV."""
import contextlib
import csv
import io
import json
import random
from pathlib import Path

import numpy as np
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

STATS = ["AP", "AP50", "AP75", "APs", "APm", "APl"]


def set_seed(seed):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def write_json(obj, path, indent=None):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=indent)


def append_csv(path, row, fields):
    path = Path(path)
    new = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore", restval="")
        if new:
            w.writeheader()
        w.writerow(row)


def encode_mask(mask):
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return rle


def det(image_id, category_id, mask, score, bbox=None):
    """Satu prediksi dalam format hasil COCO. bbox (x, y, w, h) dihitung dari mask kalau tidak diberikan."""
    rle = encode_mask(mask)
    if bbox is None:
        bbox = mask_utils.toBbox(rle).tolist()
    return {"image_id": int(image_id), "category_id": int(category_id), "segmentation": rle,
            "bbox": [float(v) for v in bbox], "score": float(score)}


def coco_evaluate(gt_json, dets):
    """Mask AP dan box AP gaya COCO, plus mask AP per kelas.
    Untuk 'segm', bbox dibuang dari prediksi supaya area dihitung dari mask (sama seperti Detectron2)."""
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(gt_json))
    metrics, per_class = {}, {}
    for iou_type in ["segm", "bbox"]:
        if not dets:
            metrics.update({f"{iou_type}_{s}": 0.0 for s in STATS})
            continue
        d = [{k: v for k, v in x.items() if not (iou_type == "segm" and k == "bbox")} for x in dets]
        with contextlib.redirect_stdout(io.StringIO()):
            ev = COCOeval(gt, gt.loadRes(d), iou_type)
            ev.evaluate()
            ev.accumulate()
            ev.summarize()
        metrics.update({f"{iou_type}_{s}": float(v) for s, v in zip(STATS, ev.stats[:6])})
        if iou_type == "segm":
            prec = ev.eval["precision"][:, :, :, 0, -1]   # IoU x recall x kelas, area=all, maxDets=100
            for k, cat_id in enumerate(ev.params.catIds):
                p = prec[:, :, k]
                p = p[p > -1]
                per_class[gt.cats[cat_id]["name"]] = float(p.mean()) if p.size else float("nan")
    return metrics, per_class


def evaluate_and_save(gt_json, dets, out_dir, split, extra=None):
    metrics, per_class = coco_evaluate(gt_json, dets)
    write_json(dets, Path(out_dir) / f"preds_{split}.json")
    result = {"split": split, **(extra or {}), **metrics, "per_class_segm_AP": per_class}
    write_json(result, Path(out_dir) / f"eval_{split}.json", indent=2)
    print(f"[{split}] mask AP={metrics['segm_AP']:.4f}  AP50={metrics['segm_AP50']:.4f}  "
          f"box AP={metrics['bbox_AP']:.4f}  ({len(dets)} prediksi)", flush=True)
    return result

In [ ]:
%%writefile train_maskrcnn.py
"""Training & prediksi Mask R-CNN R50-FPN v2 (torchvision) untuk satu fraksi data. Satu proses, satu GPU."""
import argparse
import random
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torchvision.transforms.functional as TF
from PIL import Image
from pycocotools import mask as mask_utils
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights, maskrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import masks_to_boxes

from seg_common import append_csv, coco_evaluate, det, evaluate_and_save, read_json, set_seed, write_json

warnings.filterwarnings("ignore", message="Detected call of")   # GradScaler melewati step pertama (normal di fp16)

FIELDS = ["epoch", "lr", "loss", "loss_classifier", "loss_box_reg", "loss_mask", "loss_objectness",
          "loss_rpn_box_reg", "skipped_iters", "train_time_s", "val_segm_AP", "val_segm_AP50", "val_bbox_AP",
          "saved_best", "elapsed_h", "gpu_mem_gb"]


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("mode", choices=["train", "predict"])
    p.add_argument("--tag", required=True)
    p.add_argument("--name", required=True)
    p.add_argument("--data-root", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--epochs", type=int, default=24)
    p.add_argument("--batch", type=int, default=4)
    p.add_argument("--lr", type=float, default=0.005)
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--eval-every", type=int, default=2)
    p.add_argument("--max-hours", type=float, default=0)
    p.add_argument("--amp", action=argparse.BooleanOptionalAction, default=True)
    p.add_argument("--splits", nargs="+", default=["valid", "test"])
    return p.parse_args()


class CocoInstances(torch.utils.data.Dataset):
    def __init__(self, json_path, img_dir, train):
        coco = read_json(json_path)
        self.images, self.img_dir, self.train = coco["images"], Path(img_dir), train
        self.anns = defaultdict(list)
        for a in coco["annotations"]:
            self.anns[a["image_id"]].append(a)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        im = self.images[i]
        img = TF.to_tensor(Image.open(self.img_dir / im["file_name"]).convert("RGB"))
        h, w = img.shape[1:]
        anns = self.anns[im["id"]]
        if anns:
            masks = torch.from_numpy(np.stack([
                mask_utils.decode(mask_utils.merge(mask_utils.frPyObjects(a["segmentation"], h, w))) for a in anns]))
        else:
            masks = torch.zeros((0, h, w), dtype=torch.uint8)
        labels = torch.tensor([a["category_id"] for a in anns], dtype=torch.int64)   # 1..N, 0 = background
        if self.train and random.random() < 0.5:
            img, masks = img.flip(-1), masks.flip(-1)
        keep = masks.flatten(1).amax(1) > 0
        masks, labels = masks[keep], labels[keep]
        boxes = masks_to_boxes(masks) if len(masks) else torch.zeros((0, 4))
        boxes[:, 2:] += 1                      # koordinat piksel inklusif -> eksklusif (lebar/tinggi >= 1)
        return img, {"boxes": boxes, "labels": labels, "masks": masks}


def collate(batch):
    return tuple(zip(*batch))


def build_model(num_classes, imgsz):
    # multi-scale training: sisi pendek acak 0.75x..1x imgsz, sisi panjang maks imgsz; evaluasi pakai imgsz
    sizes = tuple(sorted(set(range(int(imgsz * 0.75) // 32 * 32, imgsz + 1, 32)) | {imgsz}))
    model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1, min_size=sizes,
                                     max_size=imgsz, box_detections_per_img=100, box_score_thresh=0.001)
    model.roi_heads.box_predictor = FastRCNNPredictor(
        model.roi_heads.box_predictor.cls_score.in_features, num_classes + 1)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        model.roi_heads.mask_predictor.conv5_mask.in_channels, 256, num_classes + 1)
    return model


@torch.no_grad()
def predict_split(model, root, split, dev, amp):
    gt_json = root / "annotations" / f"{split}.json"
    images = read_json(gt_json)["images"]
    model.eval()
    dets, t_inf = [], 0.0
    for i, im in enumerate(images):
        t0 = time.time()
        img = TF.to_tensor(Image.open(root / "images" / split / im["file_name"]).convert("RGB")).to(dev)
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            out = model([img])[0]
        masks = (out["masks"][:, 0] > 0.5).cpu().numpy()
        boxes = out["boxes"].float().cpu().numpy()
        labels = out["labels"].cpu().numpy()
        scores = out["scores"].float().cpu().numpy()
        if i > 0:
            t_inf += time.time() - t0
        for m, b, c, s in zip(masks, boxes, labels, scores):
            if m.any():
                dets.append(det(im["id"], c, m, s, [b[0], b[1], b[2] - b[0], b[3] - b[1]]))
    return gt_json, dets, 1000 * t_inf / max(len(images) - 1, 1)


def train(a):
    set_seed(a.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = a.amp and dev.type == "cuda"
    root, run = Path(a.data_root), Path(a.out) / a.name
    (run / "weights").mkdir(parents=True, exist_ok=True)
    (run / "results.csv").unlink(missing_ok=True)
    write_json(vars(a), run / "args.json", indent=2)
    num_classes = len(read_json(root / "dataset_info.json")["classes"])

    ds = CocoInstances(root / "annotations" / f"train_{a.tag}.json", root / "images" / "train", train=True)
    dl = torch.utils.data.DataLoader(ds, batch_size=a.batch, shuffle=True, num_workers=a.workers,
                                     collate_fn=collate, generator=torch.Generator().manual_seed(a.seed),
                                     persistent_workers=a.workers > 0, pin_memory=dev.type == "cuda")
    model = build_model(num_classes, a.imgsz).to(dev)
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=a.lr, momentum=0.9,
                          weight_decay=1e-4)
    total = a.epochs * len(dl)
    warm = max(1, min(500, total // 10))
    milestones = (int(0.7 * total), int(0.9 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda it: min(1.0, 0.001 + 0.999 * it / warm) * 0.1 ** sum(it >= m for m in milestones))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    print(f"train_{a.tag}: {len(ds)} gambar, {len(dl)} iterasi/epoch, {total} iterasi, device={dev}, amp={amp}",
          flush=True)

    best, t_start, stopped_by_time, ep = -1.0, time.time(), False, 0
    for ep in range(1, a.epochs + 1):
        model.train()
        t0, sums, n, skipped = time.time(), defaultdict(float), 0, 0
        for imgs, tgts in dl:
            imgs = [x.to(dev, non_blocking=True) for x in imgs]
            tgts = [{k: v.to(dev, non_blocking=True) for k, v in t.items()} for t in tgts]
            with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
                losses = model(imgs, tgts)
            loss = sum(losses.values())
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1
                sched.step()
                continue
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            n += 1
            sums["loss"] += loss.item()
            for k, v in losses.items():
                sums[k] += v.item()
        row = {"epoch": ep, "lr": f"{max(g['lr'] for g in opt.param_groups):.6f}", "skipped_iters": skipped,
               "train_time_s": round(time.time() - t0, 1), **{k: round(v / max(n, 1), 5) for k, v in sums.items()}}
        stopped_by_time = a.max_hours > 0 and (time.time() - t_start) / 3600 > a.max_hours
        if ep % a.eval_every == 0 or ep == a.epochs or stopped_by_time:
            gt_json, dets, _ = predict_split(model, root, "valid", dev, amp)
            m, _ = coco_evaluate(gt_json, dets)
            row.update(val_segm_AP=round(m["segm_AP"], 5), val_segm_AP50=round(m["segm_AP50"], 5),
                       val_bbox_AP=round(m["bbox_AP"], 5))
            if m["segm_AP"] > best:
                best = m["segm_AP"]
                torch.save(model.state_dict(), run / "weights" / "best.pth")
                row["saved_best"] = 1
        row["elapsed_h"] = round((time.time() - t_start) / 3600, 4)
        row["gpu_mem_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2) if dev.type == "cuda" else 0
        append_csv(run / "results.csv", row, FIELDS)
        print(" | ".join(f"{k}={row[k]}" for k in FIELDS if k in row), flush=True)
        if stopped_by_time:
            print(f"[time_guard] melewati {a.max_hours} jam, berhenti setelah epoch {ep}", flush=True)
            break
    write_json({"tag": a.tag, "epochs_planned": a.epochs, "epochs_done": ep, "stopped_by_time": stopped_by_time,
                "best_val_segm_AP": round(best, 5), "train_hours": round((time.time() - t_start) / 3600, 4),
                "train_images": len(ds)}, run / "train_summary.json", indent=2)


def predict(a):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = a.amp and dev.type == "cuda"
    root, run = Path(a.data_root), Path(a.out) / a.name
    model = build_model(len(read_json(root / "dataset_info.json")["classes"]), a.imgsz).to(dev)
    model.load_state_dict(torch.load(run / "weights" / "best.pth", map_location=dev, weights_only=True))
    for split in a.splits:
        if not (root / "annotations" / f"{split}.json").exists():
            print(f"[{split}] tidak ada, dilewati")
            continue
        gt_json, dets, ms = predict_split(model, root, split, dev, amp)
        evaluate_and_save(gt_json, dets, run, split, {"model": "mask_rcnn", "tag": a.tag, "ms_per_image": round(ms, 2)})


if __name__ == "__main__":
    args = parse()
    train(args) if args.mode == "train" else predict(args)

## 4. Bobot pretrained dan runner

In [ ]:
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights

# Unduh bobot COCO sekali ke cache torch, supaya dua proses paralel tidak mengunduh bersamaan.
MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1.get_state_dict(progress=True)

SCRIPT = Path("train_maskrcnn.py").resolve()


def train_cmd(tag, epochs, name):
    return [sys.executable, SCRIPT, "train", "--tag", tag, "--name", name, "--data-root", DATA_ROOT, "--out", RUNS,
            "--epochs", epochs, "--batch", BATCH, "--lr", LR, "--imgsz", IMGSZ, "--workers", WORKERS,
            "--seed", SEED, "--eval-every", 1 if PILOT else EVAL_EVERY, "--max-hours", MAX_HOURS_PER_RUN]


def predict_cmd(tag, name):
    return [sys.executable, SCRIPT, "predict", "--tag", tag, "--name", name, "--data-root", DATA_ROOT,
            "--out", RUNS, "--imgsz", IMGSZ, "--splits", *EVAL_SPLITS]

In [ ]:
import re

LOG_DIR = WORK / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
GPU_LOG = WORK / "gpu_time_log.csv"
N_GPU = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPU)]
print(f"{N_GPU} GPU:", GPU_NAMES)
_lock = threading.Lock()


def job(kind, name, cmd):
    return {"kind": kind, "name": name, "cmd": [str(c) for c in cmd], "log": str(LOG_DIR / f"{name}_{kind}.log")}


def _tail(path, n=1, nbytes=4000):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - nbytes))
            text = f.read().decode("utf-8", "ignore")
    except FileNotFoundError:
        return []
    return [p.strip() for p in re.split(r"[\r\n]+", text) if p.strip()][-n:]


def _last_line(path):
    lines = _tail(path)
    return lines[-1][:170] if lines else ""


def _run_job(j, gpu):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    start, t0 = datetime.now(), time.time()
    with open(j["log"], "w", encoding="utf-8") as log:
        log.write("$ " + " ".join(j["cmd"]) + "\n")
        log.flush()
        rc = subprocess.run(j["cmd"], env=env, stdout=log, stderr=subprocess.STDOUT, cwd=WORK).returncode
    row = {"model": MODEL_KEY, "job": j["name"], "kind": j["kind"], "gpu_index": gpu,
           "gpu_name": GPU_NAMES[gpu] if gpu < N_GPU else "cpu",
           "start": start.isoformat(timespec="seconds"), "end": datetime.now().isoformat(timespec="seconds"),
           "hours": round((time.time() - t0) / 3600, 4), "returncode": rc}
    with _lock:
        pd.DataFrame([row]).to_csv(GPU_LOG, mode="a", header=not GPU_LOG.exists(), index=False)
        status = "OK" if rc == 0 else f"GAGAL (kode {rc}), lihat {j['log']}"
        print(f"[{row['end']}] {j['kind']} {j['name']} di GPU{gpu}: {row['hours']:.2f} jam, {status}", flush=True)
        if rc != 0:                                  # tampilkan traceback di output notebook, bukan hanya di file log
            print("    | " + "\n    | ".join(_tail(j["log"], 25, 12000)), flush=True)
    return rc


def run_jobs(jobs):
    """Dengan 2 GPU: job pertama di GPU0, sisanya berurutan di GPU1 (dua jalur paralel).
    Dengan 1 GPU: semua berurutan di GPU0. Setiap job = satu proses terpisah dengan satu GPU."""
    lanes = {0: jobs[:1], 1: jobs[1:]} if N_GPU >= 2 and len(jobs) > 1 else {0: jobs}
    results, started = {}, set()

    def lane(gpu, lane_jobs):
        for j in lane_jobs:
            started.add(j["name"])
            results[j["name"]] = _run_job(j, gpu)

    threads = [threading.Thread(target=lane, args=(g, js)) for g, js in lanes.items() if js]
    for t in threads:
        t.start()
    t0 = last = time.time()
    while any(t.is_alive() for t in threads):
        time.sleep(2)
        if time.time() - last >= POLL_SECONDS:
            last = time.time()
            for j in jobs:
                if j["name"] in started and j["name"] not in results:
                    print(f"  {(time.time() - t0) / 60:6.1f} mnt | {j['name']}: {_last_line(j['log'])}", flush=True)
    for t in threads:
        t.join()
    return results

## 5. Training (atau pilot)

In [ ]:
order = sorted(TAGS, key=lambda t: (t != "100", t))          # run 100% di jalur GPU0, 025 lalu 050 di GPU1
run_name = lambda tag: f"{MODEL_KEY}_{tag}"

if PILOT:
    name = f"{MODEL_KEY}_100_pilot"
    run_jobs([job("pilot", name, train_cmd("100", PILOT_EPOCHS, name))])
    hours = pd.read_csv(GPU_LOG).query("job == @name")["hours"].iloc[-1]
    per_epoch_min = hours * 60 / PILOT_EPOCHS
    print(f"\nPilot: {PILOT_EPOCHS} epoch dalam {hours * 60:.1f} menit -> ~{per_epoch_min:.1f} menit/epoch "
          f"(termasuk waktu muat model, jadi sedikit lebih besar dari aslinya)")
    print(f"Saran EPOCHS agar run 100% selesai dalam ~{TARGET_HOURS_100} jam: "
          f"{max(1, int(TARGET_HOURS_100 * 60 / per_epoch_min))}")
    print("Langkah berikutnya: isi EPOCHS, set PILOT = False, lalu Save Version -> Save & Run All.")
elif RUN_TRAINING:
    rc = run_jobs([job("train", run_name(t), train_cmd(t, EPOCHS, run_name(t))) for t in order])
    rows = []
    for t in TAGS:
        p = RUNS / run_name(t) / "train_summary.json"
        rows.append(json.loads(p.read_text()) if p.exists() else {"tag": t, "status": "GAGAL, cek log"})
    print(pd.DataFrame(rows).to_string(index=False))
else:
    found = [p for p in glob.glob(PREV_RUNS_GLOB, recursive=True)
             if all((Path(p) / run_name(t)).exists() for t in TAGS)]
    assert found, f"Folder runs berisi {[run_name(t) for t in TAGS]} tidak ditemukan di {PREV_RUNS_GLOB}"
    shutil.copytree(found[0], RUNS, dirs_exist_ok=True)
    for f in [*RUNS.glob("*/eval_*.json"), *RUNS.glob("*/preds_*.json")]:
        f.unlink()                       # hasil evaluasi versi lama dibuang; semua split dihitung ulang di bagian 6
    print("Memakai bobot dari", found[0])

## 6. Evaluasi pada valid, test, dan test_own

In [ ]:
if not PILOT:
    run_jobs([job("predict", run_name(t), predict_cmd(t, run_name(t))) for t in order
              if (RUNS / run_name(t)).exists()])

    rows, per_class = [], {}
    for t in TAGS:
        summ_p = RUNS / run_name(t) / "train_summary.json"
        summ = json.loads(summ_p.read_text()) if summ_p.exists() else {}
        for s in EVAL_SPLITS:
            p = RUNS / run_name(t) / f"eval_{s}.json"
            if not p.exists():
                continue
            r = json.loads(p.read_text())
            per_class[(t, s)] = r.pop("per_class_segm_AP")
            rows.append({"model": MODEL_LABEL, "family": FAMILY, "fraksi": int(t), "split": s,
                         **{k: round(v, 4) for k, v in r.items() if k.startswith(("segm_", "bbox_"))},
                         "ms_per_image": r.get("ms_per_image"), "train_hours": summ.get("train_hours"),
                         "epochs_done": summ.get("epochs_done"), "stopped_by_time": summ.get("stopped_by_time")})
    metrics = pd.DataFrame(rows)
    metrics.to_csv(WORK / f"metrics_{MODEL_KEY}.csv", index=False)
    pc = pd.DataFrame(per_class).round(4)
    pc.columns = [f"{t}_{s}" for t, s in pc.columns]
    pc.to_csv(WORK / f"per_class_{MODEL_KEY}.csv")

    print("Mask AP (COCO AP@[.50:.95]) per fraksi data latih:")
    print(metrics.pivot(index="fraksi", columns="split", values="segm_AP").reindex(columns=EVAL_SPLITS).to_string(), "\n")
    print("Mask AP50:")
    print(metrics.pivot(index="fraksi", columns="split", values="segm_AP50").reindex(columns=EVAL_SPLITS).to_string(), "\n")
    print("Mask AP per kelas:")
    print(pc.to_string())

## 7. Ringkasan waktu GPU dan output

In [ ]:
if GPU_LOG.exists():
    log = pd.read_csv(GPU_LOG)
    print(log.to_string(index=False))
    print("\nTotal jam-GPU per jenis job:")
    print(log.groupby("kind")["hours"].sum().round(2).to_string())

print("\nOutput penting di", WORK)
for p in sorted(WORK.glob("*.csv")):
    print("  ", p.name)
for p in sorted(RUNS.glob("*/")):
    files = [f.name for f in p.iterdir()]
    print("  ", p.relative_to(WORK), "->", ", ".join(sorted(files))[:150])

## Catatan untuk logbook dan laporan

Isi setelah notebook selesai (bagian 4 tugas: bukti proses):

- **Waktu GPU:** salin tabel dari `gpu_time_log.csv` (jam per job, GPU yang dipakai).
- **Log training asli:** `runs/<run>/results.csv` untuk setiap fraksi.
- **Keputusan:** jumlah epoch dan alasannya (hasil pilot), batch size, ukuran input, apakah ada run yang berhenti karena batas waktu (`stopped_by_time`).
- **Kendala:** error, OOM, atau hal yang diubah selama pengerjaan, beserta solusinya.
- **Bantuan AI:** bagian kode yang dibuat dengan AI dan kasus saat saran AI salah atau perlu diperbaiki (bagian 5 tugas).